In [3]:
import MLM.moire_lattice_vector_finder as mlf
import numpy as np
import pandas as pd
import time
import sys, os
from pprint import pprint
from MLM.match import run_and_filter


In [1]:
file_1 = "../moire_structures/PTO_STO/pto_tetra_monolayer.vasp"
file_2 = "../moire_structures/PTO_STO/STO_centered_unit_cell_cubic_monolayer.vasp"

In [4]:
lattice_vectors1, atom_type_arr1, dat1 = mlf.read_vasp(file_1)

In [5]:
lattice_vectors1['a']

array([3.8795519, 0.       , 0.       ])

In [7]:
lattice_vectors2, atom_type_arr2, dat2 = mlf.read_vasp(file_2)

In [8]:
mlf.angle_between(lattice_vectors1['a'],lattice_vectors1['b'])

90.0

In [37]:
results = run_and_filter(a1=lattice_vectors1['a'][0:2],a2=lattice_vectors1['b'][0:2],
                      g1=lattice_vectors2['a'][0:2],g2=lattice_vectors2['b'][0:2],
                      degmin=1.0, degmax = 45.00, degstep = 0.01,
                      dtol = 9e-3, nmin = -10, nmax = 10, goodangles = [90.0])

In [38]:
len(results)

1

In [39]:
angle=[]
A1 = []
A2 = []
delvec = []
delgood = []
norm_A1 = []
norm_A2 = []

for result in results[0]['results']:
    angle.append(result[0]['angle'])
    A1.append(result[0]['matchpoint'])
    A2.append(result[1]['matchpoint'])
    delvec.append(result[0]['delvec'])
    delgood.append(np.abs(result[2]['delgoodangle']))
    norm_A1.append(np.linalg.norm(np.array(result[0]['matchpoint'])))
    norm_A2.append(np.linalg.norm(np.array(result[1]['matchpoint'])))


In [40]:
resdf = pd.DataFrame(zip(angle,A1,A2,delvec,delgood,norm_A1, norm_A2), columns=['Theta','A1', 'A2', 'delvec', 'delang','norm_A1', 'norm_A2'])

In [41]:
len(resdf)

416

In [32]:
# Sort the DataFrame by 'degree' in ascending order and 'norm' in descending order, followed by 'vec_del' and 'delta_angle'
rsdf_sorted = resdf.sort_values(by=[ 'Theta'], ascending=[ True])



In [33]:
rsdf_sorted.head(60)

,Theta,A1,A2,delvec,delang,norm_A1,norm_A2
0,28.97,"[-38.795519, -34.9159671]","[34.9159671, -38.795519]",0.022054,0.0,52.194033,52.194033
1,28.97,"[34.9159671, -38.795519]","[-38.795519, -34.9159671]",0.022054,0.0,52.194033,52.194033
2,28.98,"[-38.795519, -34.9159671]","[34.9159671, -38.795519]",0.013939,0.0,52.194033,52.194033
3,28.98,"[34.9159671, -38.795519]","[-38.795519, -34.9159671]",0.013939,0.0,52.194033,52.194033
4,28.99,"[-38.795519, -34.9159671]","[34.9159671, -38.795519]",0.008259,0.0,52.194033,52.194033
5,28.99,"[34.9159671, -38.795519]","[-38.795519, -34.9159671]",0.008259,0.0,52.194033,52.194033
6,29.00,"[-38.795519, -34.9159671]","[34.9159671, -38.795519]",0.010399,0.0,52.194033,52.194033
7,29.00,"[34.9159671, -38.795519]","[-38.795519, -34.9159671]",0.010399,0.0,52.194033,52.194033
8,29.01,"[-38.795519, -34.9159671]","[34.9159671, -38.795519]",0.017721,0.0,52.194033,52.194033
9,29.01,"[34.9159671, -38.795519]","[-38.795519, -34.9159671]",0.017721,0.0,52.194033,52.194033


In [ ]:
# Group by 'degree' and take the first entry in each group
final_rsdf = rsdf_sorted.groupby('delvec', as_index=False).first()

In [ ]:
len(final_rsdf)

In [ ]:
# final_rsdf['num_atoms'] = np.round(1.574158*final_rsdf['norm_A1']*final_rsdf['norm_A2'])

In [ ]:
final_rsdf.sort_values(by="delvec", ascending=True).head(60)

In [42]:
tol = 0.05  # Theta‐difference tolerance for “closeness”

# NEW: filter where norm_A1 ≈ norm_A2 within 0.1
df_filtered = resdf[abs(resdf['norm_A1'] - resdf['norm_A2']) <= 0.1]

# 1) sort by Theta
df_sorted = df_filtered.sort_values('Theta').reset_index(drop=True)


keep_idxs = []           # indices of the rows we'll keep
current_group = [0]      # start with the first row in group

for i in range(1, len(df_sorted)):
    θ_prev = df_sorted.loc[i-1, 'Theta']
    θ_curr = df_sorted.loc[i,   'Theta']

    if abs(θ_curr - θ_prev) <= tol:
        # same cluster: add to current_group
        current_group.append(i)
    else:
        # new cluster starts: pick best from the old
        best = df_sorted.loc[current_group, 'delvec'].idxmin()
        keep_idxs.append(best)
        current_group = [i]

# final cluster
best = df_sorted.loc[current_group, 'delvec'].idxmin()
keep_idxs.append(best)

# select and restore original order (optional)
result = df_sorted.loc[keep_idxs].sort_index().reset_index(drop=True)

In [43]:
result.sort_values(by="norm_A1", ascending=True).head(60)

,Theta,A1,A2,delvec,delang,norm_A1,norm_A2
1,17.74,"[27.1568633, -7.7591038]","[7.7591038, 27.1568633]",0.028766,0.0,28.243564,28.243564
5,40.36,"[-27.1568633, 7.7591038]","[-7.7591038, -27.1568633]",0.028761,0.0,28.243564,28.243564
0,7.13,"[-31.0364152, -3.8795519]","[-3.8795519, 31.0364152]",0.023819,0.0,31.277947,31.277947
3,29.74,"[-27.1568633, -15.5182076]","[-15.5182076, 27.1568633]",0.023813,0.0,31.277947,31.277947
2,28.99,"[-38.795519, -34.9159671]","[34.9159671, -38.795519]",0.008259,0.0,52.194033,52.194033
4,35.02,"[-34.9159671, -38.795519]","[-38.795519, 34.9159671]",0.008087,0.0,52.194033,52.194033


In [44]:
result.to_pickle("../moire_structures/PTO_STO/tblstopto.pkl")

In [ ]:
len(result)